In [2]:
import ee
import geemap

ee.Initialize(project='peppy-gist-511817-e8')
print("Google Earth Engine initialized successfully!")

Google Earth Engine initialized successfully!


In [4]:
# 1. Define Area of Interest (AOI) in Mato Grosso / Rondônia, Brazilian Amazon
aoi = ee.Geometry.Polygon([[
    [-60.5, -10.0],
    [-59.5, -10.0],
    [-59.5, -11.0],
    [-60.5, -11.0]
]])

# 2. Define Dry Season Date Ranges (July to October)
start_2022 = '2022-07-01'
end_2022 = '2022-10-31'

start_2024 = '2024-07-01'
end_2024 = '2024-10-31'

# 3. Load Sentinel-2 Level-2A (Surface Reflectance) Collection
s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# Function to mask clouds using the S2 QA60 band (multiplying binary masks for logical AND)
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    
    # 0 means clear of clouds and cirrus
    cloud_mask = qa.bitwiseAnd(cloudBitMask).eq(0)
    cirrus_mask = qa.bitwiseAnd(cirrusBitMask).eq(0)
    
    # Multiply the binary masks together (logical AND)
    mask = cloud_mask.multiply(cirrus_mask)
    
    return image.updateMask(mask).divide(10000)

# 4. Filter and composite images for 2022 and 2024
image_2022 = (s2_collection
    .filterBounds(aoi)
    .filterDate(start_2022, end_2022)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .median()
    .clip(aoi))

image_2024 = (s2_collection
    .filterBounds(aoi)
    .filterDate(start_2024, end_2024)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .median()
    .clip(aoi))

print("Sentinel-2 imagery pipelines configured successfully!")

Sentinel-2 imagery pipelines configured successfully!


In [5]:
# 1. Calculate NDVI for 2022 and 2024 composites
# Sentinel-2 bands: B8 is NIR, B4 is Red
ndvi_2022 = image_2022.normalizedDifference(['B8', 'B4']).rename('ndvi')
ndvi_2024 = image_2024.normalizedDifference(['B8', 'B4']).rename('ndvi')

# 2. Compute NDVI difference (Drop in NDVI indicates potential deforestation/loss)
# Positive values mean vegetation loss (2022 NDVI was higher than 2024 NDVI)
ndvi_diff = ndvi_2022.subtract(ndvi_2024).rename('ndvi_diff')

# 3. Apply a threshold to isolate significant vegetation drops 
# (e.g., drop greater than 0.25 indicates clearing of primary cover vs natural seasonal variance)
deforestation_mask = ndvi_diff.gt(0.25)

# Mask out unchanged areas to focus purely on deforested pixels
deforestation_detected = ndvi_diff.updateMask(deforestation_mask)

print("NDVI baseline and change detection pipeline configured successfully!")

NDVI baseline and change detection pipeline configured successfully!


In [6]:
# 1. Filter out small noise fragments (< 0.5 ha = 50 pixels at 10m resolution)
min_cluster_pixels = 50
cleaned_deforestation = (deforestation_mask
    .selfMask()
    .connectedPixelCount(maxSize=100, eightConnected=True)
    .gte(min_cluster_pixels))

# 2. Set up interactive geemap visualization centered over our AOI
Map = geemap.Map(center=[-10.5, -60.0], zoom=9)

# Add True-Color RGB Composites for 2022 and 2024 (Bands B4=Red, B3=Green, B2=Blue)
Map.addLayer(image_2022.select(['B4', 'B3', 'B2']), {'min': 0, 'max': 0.3}, 'Sentinel-2 2022 (RGB)')
Map.addLayer(image_2024.select(['B4', 'B3', 'B2']), {'min': 0, 'max': 0.3}, 'Sentinel-2 2024 (RGB)')

# Add our filtered deforestation mask highlighted in bright red
Map.addLayer(cleaned_deforestation.selfMask(), {'palette': ['red']}, 'Deforestation (>0.5 ha)')

# Display the map object
Map

Map(center=[-10.5, -60.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…